# i+1 Story SLM — QLoRA fine-tuning on Colab GPU

Trains **Qwen3-4B-Instruct** with 4-bit QLoRA on the i+1 comprehensible-input story dataset,
then runs the full base-vs-tuned eval (deterministic validators **+** LLM judge **+** cloze
inferability) on the golden set and the held-out set, and pushes results to LangSmith.

**Budget:** $10 = ~100 compute units. Develop on **L4** (~20 h), reserve **A100** for the one
final run. Disconnect the runtime the moment a cell finishes - units burn on connection time.
See `docs/COLAB_PLAN.md` for the full spend plan.

Everything the notebook calls already exists in the repo; the CLI flags are wired.


## Step 0 - pick the GPU runtime

**Runtime -> Change runtime type -> L4 GPU** (switch to A100 only for the final full run).
Then run the cell below to confirm the GPU is attached.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Step 1 - clone the repo and install

Installs the package with its `train` extras (torch/transformers/trl/peft/datasets/accelerate)
plus `bitsandbytes` for 4-bit QLoRA (CUDA only) and the `langsmith` client for Step 6b.


In [ ]:
REPO_URL = 'https://github.com/1624252/slm.git'  # <-- your GitHub repo

import os
if not os.path.isdir('slm'):
    !git clone $REPO_URL
%cd slm
!pip -q install -e '.[train,langsmith]' bitsandbytes
print('\nInstalled.')

## Step 2 - API keys from Colab Secrets (judge + tracking)

The LLM judge and the LangSmith push read keys from the environment. **Do not paste keys into
the notebook.** Add them in Colab's **Secrets** panel (key icon, left sidebar), then this cell
loads them:

| Secret name | Value | Used for |
| --- | --- | --- |
| `OPENAI_API_KEY` | your `tfy_va_...` key | LLM judge (load-bearing) |
| `OPENAI_BASE_URL` | `https://tfy.promptlens.trilogy.com/v1` | LLM judge endpoint |
| `JUDGE_MODEL` | `claude-group/claude-sonnet-4-6` | judge model |
| `LANGSMITH_API_KEY` | your `lsv2_pt_...` key | dashboard push (Step 6b) |
| `LANGSMITH_PROJECT` | `islm-evals` (optional) | dashboard project name |

Toggle **Notebook access** on for each. Missing keys degrade gracefully: no judge key -> eval
falls back to deterministic-only; no LangSmith key -> Step 6b no-ops (local leaderboard still
records the run).


In [ ]:
from google.colab import userdata
import os

# Copy Colab secrets -> environment (islm reads these; no .env needed on Colab).
_SECRETS = ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'JUDGE_MODEL',
            'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT']
for name in _SECRETS:
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
    except Exception:
        pass  # secret not set / access not granted -> feature degrades gracefully
os.environ.setdefault('LANGSMITH_PROJECT', 'islm-evals')

print('judge key set :', bool(os.environ.get('OPENAI_API_KEY')))
print('judge model   :', os.environ.get('JUDGE_MODEL', '(default)'))
print('langsmith set :', bool(os.environ.get('LANGSMITH_API_KEY')))

## Step 3 - get the dataset (ships in the repo, gzipped)

The dataset (**144,748** spec-passing records, 2-3 target words each, en/zh/ja) ships in the repo
as gzipped splits under `data/dataset_v1/`, so `git clone` already brought it - **no regeneration
needed**. This cell just decompresses it (a few seconds).

`SUBSET` trims the **training** split to fit the $10 budget (val/test stay full). Set it to `None`
to train on all 144k (needs the A100 + most of your credit).

*(Want to rebuild from scratch instead - e.g. a different size or seed? See "Reproduce / scale" in
`docs/DATA_CARD.md`; it uses `islm.datagen.synth` + `curate --fast`.)*

In [ ]:
# Decompress the shipped dataset (already cloned with the repo).
!gunzip -kf data/dataset_v1/train.jsonl.gz data/dataset_v1/val.jsonl.gz data/dataset_v1/test.jsonl.gz
!wc -l data/dataset_v1/*.jsonl

# Trim the training split to fit the budget (val/test kept full). None = full 144k.
SUBSET = 30000
if SUBSET:
    import random
    lines = open('data/dataset_v1/train.jsonl', encoding='utf-8').read().splitlines()
    random.Random(0).shuffle(lines)
    with open('data/dataset_v1/train.jsonl', 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines[:SUBSET]) + '\n')
    print(f'train subsampled -> {SUBSET} records')
print('Dataset ready.')

## Step 4 - smoke test (Phase 0, ~3 units)

One tiny run to catch env bugs before the real burn: confirms QLoRA loads Qwen3-4B in 4-bit and
a train step + save works. `--smoke` uses tiny settings.


In [ ]:
BASE = 'Qwen/Qwen3-4B-Instruct'  # for a T4 (16 GB) fall back to 'Qwen/Qwen3-1.7B'

!python -m islm.train.sft --data data/dataset_v1 --base $BASE \
    --qlora --smoke --out outputs/colab_smoke
print('\nSmoke OK if an adapter was written to outputs/colab_smoke.')

## Step 5 - QLoRA fine-tune (aligned recipe)

Best-known recipe from the local runs, GPU-scaled: rank 32 / alpha 64, lr 2e-4, cosine schedule
with warmup + weight decay + grad clipping (wired inside `islm.train.sft`). Saves a **LoRA
adapter** (~tens of MB) - no `--merge`, since eval loads base + adapter and the 8 GB merged
model isn't needed.

Trains on `data/dataset_v1` (the split Step 3 prepared - `SUBSET` records, or full 144k). For an
even cheaper first pass, add `--epochs 1` or cap steps with `--max-steps 200`.

In [ ]:
!python -m islm.train.sft --data data/dataset_v1 --base $BASE --qlora \
    --epochs 3 --lr 2e-4 --lora-r 32 --lora-alpha 64 \
    --max-seq-len 1024 --grad-accum 8 \
    --out outputs/qwen3_4b_qlora
print('\nDone. LoRA adapter in outputs/qwen3_4b_qlora.')

## Step 6 - evaluate (base vs tuned, all criteria, tracked)

Runs the model on both eval targets. Each reports **every criterion** with base + tuned
columns: deterministic (hard-pass, OOV, <=1-new, recurrence), the 8-dim LLM judge rubric (incl.
coherence + interestingness), and cloze inferability. The judge fires automatically because
Step 2 set the key.

- **Golden eval** - the every-commit correctness gate (`evals/golden/golden.jsonl`).
- **Held-out eval** - wider coverage sweep across en/zh/ja + adversarial.


In [ ]:
# Golden set (all languages).
!python -m islm.eval.run --golden \
    --base-path $BASE --tuned-path $BASE --tuned-adapter outputs/qwen3_4b_qlora \
    --no-think --track --run-label colab-qwen3-4b-golden \
    --dataset data/dataset_v1 --epochs 3 --out evals/colab_golden

In [ ]:
# Held-out set (all languages) + adversarial.
!python -m islm.eval.run \
    --base-path $BASE --tuned-path $BASE --tuned-adapter outputs/qwen3_4b_qlora \
    --adversarial --no-think --track --run-label colab-qwen3-4b \
    --dataset data/dataset_v1 --epochs 3 --out evals/colab_heldout

In [ ]:
# Show the result tables inline.
import glob
for f in sorted(glob.glob('evals/colab_*/results*.md')):
    print('=== ' + f + ' ===')
    print(open(f, encoding='utf-8').read())

## Step 6b - push results to LangSmith (dashboard)

LangSmith runs in *augment* mode: the eval above already computed and saved every score locally
(and `--track` logged the run to `evals/results/runs.jsonl`). This cell ships those results to
your LangSmith project as experiments, and registers the golden set as a dataset, so runs are
comparable in the dashboard. It **no-ops** if `LANGSMITH_API_KEY` is not set - nothing else
depends on it.


In [ ]:
import glob, os
if os.environ.get('LANGSMITH_API_KEY'):
    # One experiment per per-language results JSON (golden + held-out).
    for f in sorted(glob.glob('evals/colab_*/results*.json')):
        exp = os.path.basename(os.path.dirname(f)) + '-' + os.path.basename(f)[:-5]
        !python -m islm.eval.langsmith_sync results --results "$f" --experiment "$exp"
    # Register the golden set as a LangSmith dataset (idempotent).
    !python -m islm.eval.langsmith_sync golden --golden evals/golden/golden.jsonl
    print('\nPushed to LangSmith project:', os.environ.get('LANGSMITH_PROJECT'))
else:
    print('LANGSMITH_API_KEY not set - skipping push. Local runs.jsonl still has the run.')

## Step 7 - download the adapter + results

Zips the trained LoRA adapter (~tens of MB), the eval outputs, and the updated `runs.jsonl`
leaderboard so you can pull them locally and record the run in `evals/RESULTS_LOG.md`. The 8 GB
merged model is not built, so this download stays small.

In [ ]:
!zip -qr colab_artifacts.zip outputs/qwen3_4b_qlora evals/colab_golden evals/colab_heldout evals/results/runs.jsonl 2>/dev/null
from google.colab import files
files.download('colab_artifacts.zip')

## After Colab (local, free)

1. Unzip `colab_artifacts.zip` into the repo.
2. Record the run in `evals/RESULTS_LOG.md` (config + base->tuned deltas) and update
   `evals/LEADERBOARD.md`.
3. Commit (data stays git-ignored; only results/adapter-summaries are tracked).

**Unit discipline:** disconnect the runtime now. Sweep small, run big once. If 4B is tight on
VRAM, drop `--max-seq-len` before dropping the model; last resort switch `BASE` to
`Qwen/Qwen3-1.7B`.
